<a href="https://colab.research.google.com/github/Feone05/YOLOv11/blob/main/YOLOV11_FINAL_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv11 Thesis Notebook
**Three pipelines:** Raw | Zero-DCE | Zero-3DCE

Run all cells top to bottom. Never skip a cell within a section.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────
!pip install roboflow ultralytics pytorch-msssim -q
print('Dependencies installed ✓')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 70.0 MB/s eta 0:00:00
Dependencies installed ✓


In [ ]:
# ── Download dataset and create 3 isolated copies ────────────────
from roboflow import Roboflow
import shutil, os

rf      = Roboflow(api_key="s3tKV1FBAKtkaJ4hAwYw")
project = rf.workspace("thesis-fwlrq").project("annotations-okdeo")
dataset = project.version(1).download("yolov11")
src     = dataset.location
print(f'Downloaded to: {src}')

for dest in [
    '/content/datasets/Raw_Dataset',
    '/content/datasets/ZeroDCE_Dataset',
    '/content/datasets/Zero3DCE_Dataset',
]:
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    print(f'Created: {dest}')

print('\n✅ All datasets ready.')


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Annotations-1 in yolov11:: 100%|██████████| 1026/1026 [00:00<00:00, 2617.91it/s]


Downloaded to: /content/Annotations-1
Created: /content/datasets/Raw_Dataset
Created: /content/datasets/ZeroDCE_Dataset
Created: /content/datasets/Zero3DCE_Dataset

✅ All datasets ready.


## Pipeline 1: No Enhancement (Raw) + YOLOv11

In [ ]:
%cd /content
!yolo task=detect mode=train \
    model=yolo11s.pt \
    data=/content/datasets/Raw_Dataset/data.yaml \
    epochs=100 \
    imgsz=640 \
    plots=True \
    project=/content/runs/thesis \
    name=raw_experiment


/content
New https://pypi.org/project/ultralytics/8.4.64 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/Raw_Dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, mul

## Pipeline 2: Zero-DCE Enhancement + YOLOv11

In [ ]:
import os

# Clone Zero-DCE repo if needed
if not os.path.exists('/content/Zero-DCE/Zero-DCE_code/model.py'):
    !git clone https://github.com/Li-Chongyi/Zero-DCE.git /content/Zero-DCE
print('Zero-DCE repo ready ✓')


Cloning into '/content/Zero-DCE'...
remote: Enumerating objects: 236, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 236 (delta 30), reused 28 (delta 28), pack-reused 197 (from 1)
Receiving objects: 100% (236/236), 30.89 MiB | 17.66 MiB/s, done.
Resolving deltas: 100% (43/43), done.
Zero-DCE repo ready ✓


In [ ]:
import importlib.util, sys, os

for key in list(sys.modules.keys()):
    if key in ('model', 'dataloader', 'Myloss'):
        del sys.modules[key]

BASE = '/content/Zero-DCE/Zero-DCE_code'

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

model_mod = load_module('model', f'{BASE}/model.py')
from model import enhance_net_nopool
print('Zero-DCE model loaded from:', model_mod.__file__)


Zero-DCE model loaded from: /content/Zero-DCE/Zero-DCE_code/model.py


In [ ]:
import torch, glob
from torchvision import transforms
from PIL import Image
from model import enhance_net_nopool

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load pretrained Zero-DCE weights
ZERODCE_WEIGHTS = '/content/Zero-DCE/Zero-DCE_code/snapshots/Epoch99.pth'
if not os.path.exists(ZERODCE_WEIGHTS):
    # Use the weights bundled with the repo if available
    snaps = glob.glob('/content/Zero-DCE/Zero-DCE_code/snapshots/*.pth')
    if snaps:
        ZERODCE_WEIGHTS = sorted(snaps)[-1]
    else:
        raise FileNotFoundError('No Zero-DCE weights found. Check /content/Zero-DCE/Zero-DCE_code/snapshots/')

zerodce_net = enhance_net_nopool().to(device)
zerodce_net.load_state_dict(torch.load(ZERODCE_WEIGHTS, map_location=device))
zerodce_net.eval()
print(f'Zero-DCE weights loaded from: {ZERODCE_WEIGHTS}')

def enhance_zerodce(img_path):
    img            = Image.open(img_path).convert('RGB')
    orig_w, orig_h = img.size
    tf             = transforms.Compose([transforms.Resize((512,512)), transforms.ToTensor()])
    t              = tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        result   = zerodce_net(t)
        enhanced = result[-2]  # second to last is the enhanced image
    out = enhanced.squeeze(0).cpu().clamp(0, 1)
    return transforms.ToPILImage()(out).resize((orig_w, orig_h), Image.LANCZOS)

# Enhance all splits
for split in ['train', 'valid', 'test']:
    img_dir = f'/content/datasets/ZeroDCE_Dataset/{split}/images'
    if not os.path.exists(img_dir):
        print(f'  Skipping {split} — not found')
        continue
    imgs = glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png')
    print(f'Enhancing {len(imgs)} {split} images with Zero-DCE...')
    for idx, fpath in enumerate(imgs):
        enhance_zerodce(fpath).save(fpath)
        if (idx+1) % 50 == 0 or (idx+1) == len(imgs):
            print(f'  {idx+1}/{len(imgs)}')
    print(f'  {split} done ✓')

print('\n✅ Zero-DCE enhancement complete.')


Zero-DCE weights loaded from: /content/Zero-DCE/Zero-DCE_code/snapshots/Epoch99.pth
Enhancing 361 train images with Zero-DCE...
  50/361
  100/361
  150/361
  200/361
  250/361
  300/361
  350/361
  361/361
  train done ✓
Enhancing 50 valid images with Zero-DCE...
  50/50
  valid done ✓
Enhancing 100 test images with Zero-DCE...
  50/100
  100/100
  test done ✓

✅ Zero-DCE enhancement complete.


In [ ]:
%cd /content
!yolo task=detect mode=train \
    model=yolo11s.pt \
    data=/content/datasets/ZeroDCE_Dataset/data.yaml \
    epochs=100 \
    imgsz=640 \
    plots=True \
    project=/content/runs/thesis \
    name=zerodce_experiment


/content
New https://pypi.org/project/ultralytics/8.4.64 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/ZeroDCE_Dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0,

## Pipeline 3: Zero-3DCE Enhancement + YOLOv11

> **Prerequisite:** Run `Zero3DCE_Train.ipynb` first to produce `zero3dce_final.pth` in your Google Drive.

In [ ]:
# ── Mount Drive and set checkpoint path ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR   = '/content/drive/MyDrive/zero3dce/checkpoints'
final_path = os.path.join(SAVE_DIR, 'zero3dce_final.pth')

if not os.path.exists(final_path):
    raise FileNotFoundError(
        f'Checkpoint not found at {final_path}.\n'
        'Run Zero3DCE_Train.ipynb first to train and save the model.'
    )
print(f'Checkpoint found: {final_path} ✓')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint found: /content/drive/MyDrive/zero3dce/checkpoints/zero3dce_final.pth ✓


In [ ]:
# ── Zero-3DCE Architecture (must match Zero3DCE_Train.ipynb) ─────
import torch
import torch.nn as nn
import torch.nn.functional as F

class SepConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_channels, in_channels, kernel_size=kernel_size,
                                   padding=padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn  = nn.BatchNorm3d(out_channels)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.pointwise(self.depthwise(x))))

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv    = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg   = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = SepConv3d(in_ch, out_ch)
        self.attn = SpatialAttention()
        self.pool = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
    def forward(self, x):
        x = self.attn(self.conv(x))
        return self.pool(x), x

class Zero3DCE(nn.Module):
    def __init__(self, num_iterations=8):
        super().__init__()
        self.num_iterations = num_iterations
        self.enc1       = EncoderBlock(3,  32)
        self.enc2       = EncoderBlock(32, 32)
        self.enc3       = EncoderBlock(32, 32)
        self.bottleneck = SepConv3d(32, 32)
        self.dec1       = SepConv3d(64, 32)
        self.dec2       = SepConv3d(64, 32)
        self.dec3       = SepConv3d(64, 24)
        self.up         = nn.Upsample(scale_factor=(1,2,2), mode='trilinear', align_corners=False)
        self.tanh       = nn.Tanh()
    def apply_curve(self, frame, alpha):
        return torch.clamp(frame + alpha * frame * (frame - 1), 0, 1)
    def forward_with_alphas(self, x):
        x1, skip1 = self.enc1(x)
        x2, skip2 = self.enc2(x1)
        x3, skip3 = self.enc3(x2)
        xb = self.bottleneck(x3)
        xb = self.dec1(torch.cat([self.up(xb), skip3], dim=1))
        xb = self.dec2(torch.cat([self.up(xb), skip2], dim=1))
        xb = self.dec3(torch.cat([self.up(xb), skip1], dim=1))
        alpha_maps = self.tanh(xb)
        enhanced   = x
        for i in range(self.num_iterations):
            g = (i % 3) * 8
            enhanced = self.apply_curve(enhanced, alpha_maps[:, g:g+3])
        return enhanced, alpha_maps

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Architecture defined ✓  Device: {device}')


Architecture defined ✓  Device: cuda


In [ ]:
# ── Load checkpoint and enhance Zero3DCE_Dataset ─────────────────
import glob
from torchvision import transforms
from PIL import Image

model_inf = Zero3DCE().to(device)
state     = torch.load(final_path, map_location=device)
model_inf.load_state_dict(
    state['model'] if isinstance(state, dict) and 'model' in state else state
)
model_inf.eval()
print(f'Loaded: {final_path}')

def enhance_zero3dce(img_path, blend=0.65):
    img            = Image.open(img_path).convert('RGB')
    orig_w, orig_h = img.size
    tf             = transforms.Compose([transforms.Resize((256,256)), transforms.ToTensor()])
    t              = tf(img).unsqueeze(0)
    clip           = torch.stack([t, t], dim=2).to(device)
    with torch.no_grad():
        enhanced, _ = model_inf.forward_with_alphas(clip)
    out          = enhanced[0, :, 0].cpu().clamp(0, 1)
    enhanced_img = transforms.ToPILImage()(out).resize((orig_w, orig_h), Image.LANCZOS)
    # Blend: 65% enhanced + 35% original — reduces overexposure on bright CCTV zones
    return Image.blend(img.resize((orig_w, orig_h)), enhanced_img, alpha=blend)

for split in ['train', 'valid', 'test']:
    img_dir = f'/content/datasets/Zero3DCE_Dataset/{split}/images'
    if not os.path.exists(img_dir):
        print(f'  Skipping {split} — not found')
        continue
    imgs = glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png')
    print(f'Enhancing {len(imgs)} {split} images...')
    for idx, fpath in enumerate(imgs):
        enhance_zero3dce(fpath).save(fpath)
        if (idx+1) % 50 == 0 or (idx+1) == len(imgs):
            print(f'  {idx+1}/{len(imgs)}')
    print(f'  {split} done ✓')

print('\n✅ Zero-3DCE enhancement complete. Ready for YOLO training.')

Loaded: /content/drive/MyDrive/zero3dce/checkpoints/zero3dce_final.pth
Enhancing 361 train images...
  50/361
  100/361
  150/361
  200/361
  250/361
  300/361
  350/361
  361/361
  train done ✓
Enhancing 50 valid images...
  50/50
  valid done ✓
Enhancing 100 test images...
  50/100
  100/100
  test done ✓

✅ Zero-3DCE enhancement complete. Ready for YOLO training.


In [ ]:
from roboflow import Roboflow
import shutil, os

rf      = Roboflow(api_key="s3tKV1FBAKtkaJ4hAwYw")
project = rf.workspace("thesis-fwlrq").project("annotations-okdeo")
dataset = project.version(1).download("yolov11")

src  = dataset.location
dest = '/content/datasets/Zero3DCE_Dataset'
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree(src, dest)
print('Zero3DCE_Dataset reset to raw images ✓')

loading Roboflow workspace...
loading Roboflow project...
Zero3DCE_Dataset reset to raw images ✓


In [ ]:
%cd /content
!yolo task=detect mode=train \
    model=yolo11s.pt \
    data=/content/datasets/Zero3DCE_Dataset/data.yaml \
    epochs=100 \
    imgsz=640 \
    plots=True \
    project=/content/runs/thesis \
    name=zero3dce_experiment


/content
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/Zero3DCE_Dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

DEST = '/content/drive/MyDrive/thesis_exports'
os.makedirs(DEST, exist_ok=True)

print('Zipping and saving to Drive...')

shutil.make_archive(f'{DEST}/Raw_Dataset',      'zip', '/content/datasets/Raw_Dataset')
print('Raw_Dataset.zip ✓')

shutil.make_archive(f'{DEST}/ZeroDCE_Dataset',  'zip', '/content/datasets/ZeroDCE_Dataset')
print('ZeroDCE_Dataset.zip ✓')

shutil.make_archive(f'{DEST}/Zero3DCE_Dataset', 'zip', '/content/datasets/Zero3DCE_Dataset')
print('Zero3DCE_Dataset.zip ✓')

shutil.make_archive(f'{DEST}/raw_results',      'zip', '/content/runs/thesis/raw_experiment')
print('raw_results.zip ✓')

shutil.make_archive(f'{DEST}/zerodce_results',  'zip', '/content/runs/thesis/zerodce_experiment')
print('zerodce_results.zip ✓')

shutil.make_archive(f'{DEST}/zero3dce_results', 'zip', '/content/runs/thesis/zero3dce_experiment')
print('zero3dce_results.zip ✓')

shutil.copy('/content/runs/thesis/raw_experiment/weights/best.pt',      f'{DEST}/raw_best.pt')
shutil.copy('/content/runs/thesis/zerodce_experiment/weights/best.pt',  f'{DEST}/zerodce_best.pt')
shutil.copy('/content/runs/thesis/zero3dce_experiment/weights/best.pt', f'{DEST}/zero3dce_best.pt')
print('All weights ✓')

print(f'\n✅ Everything saved to Google Drive at: {DEST}')
print('Download from drive.google.com → thesis_exports folder')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zipping and saving to Drive...
Raw_Dataset.zip ✓
ZeroDCE_Dataset.zip ✓
Zero3DCE_Dataset.zip ✓
raw_results.zip ✓
zerodce_results.zip ✓
zero3dce_results.zip ✓
All weights ✓

✅ Everything saved to Google Drive at: /content/drive/MyDrive/thesis_exports
Download from drive.google.com → thesis_exports folder
